# PM100 dataset inspection

It produces an auditable clean PM100 table and chronological 5,000-job debug subset.

## Phase Decisions

- Power: `node_power_consumption` is **whole-job input power in watts**, nominally sampled every **20 seconds**. It must not be multiplied by node count and CPU/memory power must not be added to it.
- Initial cohort: `COMPLETED`, anonymized partition `1`, positive exact runtime, consistent node request/allocation, usable power coverage, and no observed node-time sharing.
- Boundary samples: accept at most one 20-second discrepancy between runtime and trace duration.
- Non-completed and ambiguous jobs: ignored.

In [9]:
from collections import OrderedDict
from hashlib import md5, sha256
from html import escape
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

POWER_SAMPLE_SECONDS = 20
SELECTED_PARTITION = "1"
DEBUG_JOB_COUNT = 5_000
WRITE_OUTPUTS = True


def display_bar_table(series, title, color="#177E89"):
    """Dependency-free horizontal bars for compact notebook EDA."""
    series = series.copy()
    maximum = max(float(series.max()), 1.0)
    rows = []
    for label, value in series.items():
        width = 100 * float(value) / maximum
        rows.append(
            "<tr>"
            f"<td style='padding:3px 10px 3px 0'>{escape(str(label))}</td>"
            f"<td style='min-width:260px'><div style='height:14px;width:{width:.2f}%;"
            f"background:{color};border-radius:3px'></div></td>"
            f"<td style='text-align:right;padding-left:10px'>{int(value):,}</td>"
            "</tr>"
        )
    display(HTML(f"<h4>{escape(title)}</h4><table>{''.join(rows)}</table>"))


DATA_PATH = Path("data/job_table.parquet")
PROJECT_ROOT = DATA_PATH.parent.parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
print(f"Input: {DATA_PATH}")
print(f"Outputs: {OUTPUT_DIR}")

Input: data/job_table.parquet
Outputs: data/processed


## 1. Load

In [10]:
parquet_file = pq.ParquetFile(DATA_PATH)
table = pq.read_table(DATA_PATH)
df = table.to_pandas(ignore_metadata=True)

source_summary = pd.DataFrame(
    [
        ("path", str(DATA_PATH)),
        ("size_bytes", DATA_PATH.stat().st_size),
        ("rows", len(df)),
        ("columns", len(df.columns)),
        ("row_groups", parquet_file.metadata.num_row_groups),
    ],
    columns=["property", "value"],
)
display(source_summary)
display(df.head(1))

,property,value
0,path,data/job_table.parquet
1,size_bytes,287154521
2,rows,231238
3,columns,35
4,row_groups,1


,cores_alloc_layout,cores_allocated,cores_per_task,derived_ec,eligible_time,end_time,group_id,job_id,job_state,nodes,num_cores_req,num_cores_alloc,num_nodes_req,num_nodes_alloc,num_tasks,partition,priority,qos,req_nodes,req_switch,run_time,shared,start_time,state_reason,submit_time,threads_per_core,time_limit,num_gpus_req,num_gpus_alloc,mem_req,mem_alloc,user_id,node_power_consumption,mem_power_consumption,cpu_power_consumption
0,"{900: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,...","{900: 128, 915: 128, 902: 128, 901: 128, 904: 128, 903: 128, 906: 128, 905: 128, 908: 128, 907: ...",4,1:0,2020-05-31 22:09:29+00:00,2020-05-31 22:21:33+00:00,25200,2913594,CANCELLED,"[900, 901, 902, 903, 904, 905, 906, 907, 908, 909, 910, 911, 912, 913, 914, 915]",256,2048,16,16,64.000,1,330603,1,NaN,0,723,0,2020-05-31 22:09:30+00:00,None,2020-05-31 22:09:29+00:00,NaN,270,64,64,475,3800,310,"[7970, 8450, 8460, 8470, 7440, 8470, 8460, 8470, 7910, 8480, 7920, 8430, 7940, 8440, 8480, 8490,...","[418, 724, 724, 678, 556, 654, 606, 600, 600, 488, 606, 446, 592, 566, 602, 560, 594, 610, 590, ...","[948, 1628, 1650, 1544, 1260, 1532, 1418, 1700, 1710, 1396, 1676, 1302, 1710, 1360, 1692, 1612, ..."


## 2. Dataset overview, missingness, states, and partitions

List-valued columns are summarized by null and empty lists rather than coerced to text.
The dominant partition is selected after showing both its scale and quality metrics.

In [11]:
LIST_COLUMNS = ["nodes", "node_power_consumption", "cpu_power_consumption", "mem_power_consumption"]
overview_rows = []
for column in df.columns:
    if column in LIST_COLUMNS:
        lengths = df[column].map(len)
        distinct = "not computed for nested lists"
        empty = int(lengths.eq(0).sum())
    else:
        distinct = int(df[column].nunique(dropna=True))
        empty = "n/a"
    overview_rows.append(
        {
            "field": column,
            "pandas_dtype": str(df[column].dtype),
            "missing": int(df[column].isna().sum()),
            "empty_lists": empty,
            "distinct_non_null": distinct,
        }
    )

column_overview = pd.DataFrame(overview_rows)
duplicate_job_ids = int(df["job_id"].duplicated(keep=False).sum())
assert duplicate_job_ids == 0

state_counts = df["job_state"].value_counts(dropna=False)
partition_counts = df["partition"].value_counts(dropna=False)
display(column_overview)
display_bar_table(state_counts, "Jobs by terminal state")
display_bar_table(partition_counts, "Jobs by anonymized partition", color="#5C4D7D")


,field,pandas_dtype,missing,empty_lists,distinct_non_null
0,cores_alloc_layout,str,0,n/a,56157
1,cores_allocated,str,0,n/a,51882
2,cores_per_task,int64,0,n/a,51
3,derived_ec,str,0,n/a,125
4,eligible_time,"datetime64[us, UTC]",0,n/a,137909
5,end_time,"datetime64[us, UTC]",0,n/a,167969
6,group_id,int64,0,n/a,5
7,job_id,int64,0,n/a,231238
8,job_state,str,0,n/a,6
9,nodes,object,0,0,not computed for nested lists


COMPLETED,,"180,310"
FAILED,,"29,840"
CANCELLED,,"11,218"
TIMEOUT,,"8,689"
OUT_OF_MEMORY,,"1,011"
NODE_FAIL,,170


1,,"228,276"
0,,"2,657"
2,,305


## 3. Timestamp and runtime validation

The required invariant is `submit_time <= eligible_time <= start_time <= end_time`.
`eligible_time` would fall back to `submit_time` only when missing; none are missing here,
so the anomalous epoch value is excluded instead of silently replaced. Runtime tolerance
is zero seconds because the source proves exact equality for every row.

In [12]:
TIMESTAMP_COLUMNS = ["submit_time", "eligible_time", "start_time", "end_time"]
timestamp_summary = pd.DataFrame(
    {
        "dtype": df[TIMESTAMP_COLUMNS].dtypes.astype(str),
        "missing": df[TIMESTAMP_COLUMNS].isna().sum(),
        "minimum": df[TIMESTAMP_COLUMNS].min(),
        "maximum": df[TIMESTAMP_COLUMNS].max(),
    }
)

df["elapsed_runtime_s"] = (df["end_time"] - df["start_time"]).dt.total_seconds().astype("int64")
df["runtime_residual_s"] = df["run_time"] - df["elapsed_runtime_s"]

time_issue_masks = OrderedDict(
    [
        ("missing_timestamp", df[TIMESTAMP_COLUMNS].isna().any(axis=1)),
        ("submit_after_eligible", df["submit_time"].gt(df["eligible_time"])),
        ("eligible_after_start", df["eligible_time"].gt(df["start_time"])),
        ("start_after_end", df["start_time"].gt(df["end_time"])),
        ("nonpositive_execution_interval", df["elapsed_runtime_s"].le(0)),
    ]
)
timestamp_checks = pd.DataFrame(
    [{"check": name, "failures": int(mask.sum())} for name, mask in time_issue_masks.items()]
)

timestamp_problem = np.logical_or.reduce(list(time_issue_masks.values()))
display(timestamp_summary)
display(timestamp_checks)
display(
    df.loc[
        timestamp_problem,
        ["job_id", "job_state", *TIMESTAMP_COLUMNS, "run_time", "elapsed_runtime_s"],
    ].head(12)
)

runtime_by_state = (
    df.assign(
        runtime_exact=df["runtime_residual_s"].eq(0),
        runtime_positive=df["run_time"].gt(0),
    )
    .groupby("job_state")
    .agg(
        jobs=("job_id", "size"),
        exact_end_minus_start=("runtime_exact", "sum"),
        positive_runtime=("runtime_positive", "sum"),
        min_runtime_s=("run_time", "min"),
        median_runtime_s=("run_time", "median"),
        max_runtime_s=("run_time", "max"),
    )
)
display(runtime_by_state)

assert df["runtime_residual_s"].eq(0).all()

,dtype,missing,minimum,maximum
submit_time,"datetime64[us, UTC]",0,2020-05-05 15:55:59+00:00,2020-10-13 04:44:57+00:00
eligible_time,"datetime64[us, UTC]",0,1970-01-01 00:00:00+00:00,2020-10-13 04:44:57+00:00
start_time,"datetime64[us, UTC]",0,2020-05-05 15:56:00+00:00,2020-10-13 04:46:59+00:00
end_time,"datetime64[us, UTC]",0,2020-05-06 08:52:54+00:00,2020-10-13 05:21:55+00:00


,check,failures
0,missing_timestamp,0
1,submit_after_eligible,1
2,eligible_after_start,0
3,start_after_end,0
4,nonpositive_execution_interval,66


,job_id,job_state,submit_time,eligible_time,start_time,end_time,run_time,elapsed_runtime_s
8371,3095598,COMPLETED,2020-05-06 12:50:57+00:00,2020-05-06 12:50:57+00:00,2020-05-06 12:51:00+00:00,2020-05-06 12:51:00+00:00,0,0
8608,825321,COMPLETED,2020-05-06 11:43:20+00:00,2020-05-06 11:43:20+00:00,2020-05-06 11:45:20+00:00,2020-05-06 11:45:20+00:00,0,0
11877,5834167,COMPLETED,2020-05-25 13:38:39+00:00,2020-05-25 13:38:39+00:00,2020-05-25 13:38:40+00:00,2020-05-25 13:38:40+00:00,0,0
15143,5709882,COMPLETED,2020-05-08 22:30:58+00:00,2020-05-08 22:30:58+00:00,2020-05-08 22:31:00+00:00,2020-05-08 22:31:00+00:00,0,0
15238,5559390,FAILED,2020-05-08 22:26:38+00:00,2020-05-08 22:26:38+00:00,2020-05-08 22:26:40+00:00,2020-05-08 22:26:40+00:00,0,0
15380,1222258,FAILED,2020-05-08 22:26:38+00:00,2020-05-08 22:26:38+00:00,2020-05-08 22:26:40+00:00,2020-05-08 22:26:40+00:00,0,0
15718,563627,FAILED,2020-05-08 22:17:58+00:00,2020-05-08 22:17:58+00:00,2020-05-08 22:18:00+00:00,2020-05-08 22:18:00+00:00,0,0
15917,3757913,FAILED,2020-05-08 22:08:18+00:00,2020-05-08 22:08:18+00:00,2020-05-08 22:08:20+00:00,2020-05-08 22:08:20+00:00,0,0
26426,4710232,CANCELLED,2020-06-12 15:49:20+00:00,2020-06-12 15:49:20+00:00,2020-06-12 15:49:20+00:00,2020-06-12 15:49:20+00:00,0,0
61544,6197968,COMPLETED,2020-06-17 13:23:37+00:00,2020-06-17 13:23:37+00:00,2020-06-17 13:23:40+00:00,2020-06-17 13:23:40+00:00,0,0


,jobs,exact_end_minus_start,positive_runtime,min_runtime_s,median_runtime_s,max_runtime_s
job_state,,,,,,
CANCELLED,11218,11218,11214,0,350.000,86231
COMPLETED,180310,180310,180263,0,190.000,86395
FAILED,29840,29840,29825,0,56.000,86330
NODE_FAIL,170,170,170,19,"27,695.000",71735
OUT_OF_MEMORY,1011,1011,1011,2,327.000,85824
TIMEOUT,8689,8689,8689,60,"7,507.000",125311


## 4. Power profiles: unit, aggregation, cadence, and completeness

The official generation code resolves the questions that the local feature table leaves open:

1. Each sample is a **power** measurement in **W**, not energy.
2. Node readings are grouped by timestamp and summed over every allocated node and both power sockets.
3. The resulting node trace is therefore the complete job aggregate.
4. The nominal interval is **20 seconds**.

The arrays do not retain individual sample timestamps. A trace is accepted when
`abs(run_time - 20 * sample_count) <= 20` seconds. This accommodates alignment and one
partial boundary interval without concealing larger gaps.

In [13]:
POWER_COLUMNS = ["node_power_consumption", "cpu_power_consumption", "mem_power_consumption"]
power_summary_rows = []

for column in POWER_COLUMNS:
    arrow_lists = table[column].combine_chunks()
    lengths = pc.list_value_length(arrow_lists).to_numpy(zero_copy_only=False)
    flat = pc.list_flatten(arrow_lists)
    power_summary_rows.append(
        {
            "profile": column,
            "null_lists": int(arrow_lists.null_count),
            "empty_lists": int(np.sum(lengths == 0)),
            "total_samples": len(flat),
            "negative_samples": int(pc.sum(pc.less(flat, 0)).as_py()),
            "zero_samples": int(pc.sum(pc.equal(flat, 0)).as_py()),
            "minimum_W": int(pc.min(flat).as_py()),
            "median_length": float(np.median(lengths)),
            "p95_length": float(np.quantile(lengths, 0.95)),
            "maximum_W": int(pc.max(flat).as_py()),
        }
    )
    df[f"{column}_samples"] = lengths

power_summary = pd.DataFrame(power_summary_rows).set_index("profile")
display(power_summary)

df["nominal_node_profile_s"] = POWER_SAMPLE_SECONDS * df["node_power_consumption_samples"]
df["node_profile_delta_s"] = df["nominal_node_profile_s"] - df["run_time"]
df["node_profile_aligned"] = df["node_profile_delta_s"].abs().le(POWER_SAMPLE_SECONDS)

cadence_summary = pd.DataFrame(
    {
        "jobs": [len(df)],
        "exact_ceil_sample_count": [
            int(
                df["node_power_consumption_samples"].eq(
                    np.ceil(df["run_time"].clip(lower=0) / POWER_SAMPLE_SECONDS).astype("int64")
                ).sum()
            )
        ],
        "within_one_20s_interval": [int(df["node_profile_aligned"].sum())],
        "larger_gap_or_excess": [int((~df["node_profile_aligned"]).sum())],
    }
)
display(cadence_summary)
display(df["node_profile_delta_s"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).to_frame())

df["node_power_mean_W"] = df["node_power_consumption"].map(np.mean)
df["node_power_min_W"] = df["node_power_consumption"].map(np.min)
df["node_power_max_W"] = df["node_power_consumption"].map(np.max)

power_scaling = (
    df.groupby("num_nodes_alloc")
    .agg(
        jobs=("job_id", "size"),
        median_whole_job_W=("node_power_mean_W", "median"),
    )
    .sort_index()
)
power_scaling["median_W_per_allocated_node"] = (
    power_scaling["median_whole_job_W"] / power_scaling.index.to_numpy(dtype=float)
)
display(power_scaling.head(20))

same_component_lengths = (
    df["node_power_consumption_samples"].eq(df["cpu_power_consumption_samples"])
    & df["node_power_consumption_samples"].eq(df["mem_power_consumption_samples"])
)

,null_lists,empty_lists,total_samples,negative_samples,zero_samples,minimum_W,median_length,p95_length,maximum_W
profile,,,,,,,,,
node_power_consumption,0,0,65173805,0,0,20,10.000,"2,159.000",438930
cpu_power_consumption,0,2867,65724638,0,0,8,10.000,"2,161.000",79568
mem_power_consumption,0,2867,65724639,0,0,18,10.000,"2,161.000",24854


,jobs,exact_ceil_sample_count,within_one_20s_interval,larger_gap_or_excess
0,231238,143760,208794,22444


,node_profile_delta_s
count,"231,238.000"
mean,-348.618
std,"3,149.537"
min,"-83,036.000"
1%,"-8,973.150"
5%,-294.000
50%,3.000
95%,19.000
99%,19.000
max,20.000


,jobs,median_whole_job_W,median_W_per_allocated_node
num_nodes_alloc,,,
1,166725,690.000,690.000
2,20358,"1,398.333",699.167
3,3012,"2,300.000",766.667
4,10377,"2,859.237",714.809
5,735,"3,760.000",752.000
6,1323,"4,364.149",727.358
7,167,"5,410.000",772.857
8,5790,"5,544.451",693.056
9,1667,"6,180.000",686.667


## 5. State and missing-power policy

Only successful completions define uncensored runtime and energy targets for the first model.
Every other terminal state is ignored rather than relabelled.

In [14]:
STATE_DECISIONS = {
    "COMPLETED": ("include when all other checks pass", "uncensored successful runtime and power target"),
    "FAILED": ("ignore", "failed execution; runtime/power may be partial or abnormal"),
    "CANCELLED": ("ignore", "user/system cancellation produces a censored execution"),
    "TIMEOUT": ("ignore", "wall-time censoring; useful only in a later censored model"),
    "OUT_OF_MEMORY": ("ignore", "resource failure and atypical termination"),
    "NODE_FAIL": ("ignore", "hardware/system failure contaminates normal-job targets"),
}

state_policy = state_counts.rename("jobs").to_frame()
state_policy["decision"] = state_policy.index.map(lambda state: STATE_DECISIONS[state][0])
state_policy["reason"] = state_policy.index.map(lambda state: STATE_DECISIONS[state][1])
display(state_policy)

,jobs,decision,reason
job_state,,,
COMPLETED,180310,include when all other checks pass,uncensored successful runtime and power target
FAILED,29840,ignore,failed execution; runtime/power may be partial or abnormal
CANCELLED,11218,ignore,user/system cancellation produces a censored execution
TIMEOUT,8689,ignore,wall-time censoring; useful only in a later censored model
OUT_OF_MEMORY,1011,ignore,resource failure and atypical termination
NODE_FAIL,170,ignore,hardware/system failure contaminates normal-job targets


## 6. Cleaning filter

A row is kept only if all declared rules pass; every other row is ignored. The aggregate
filter ledger records how many rows each stage removes.

In [15]:
# Resource-derived columns used by the cleaning rules. Keeping them in this
# cell makes the dependency explicit and prevents stale-kernel KeyErrors.
df["nodes_count"] = df["nodes"].map(len)
df["unique_nodes_count"] = df["nodes"].map(lambda nodes: len(set(nodes)))

resource_checks = pd.DataFrame(
    [
        ("empty nodes list", int(df["nodes_count"].eq(0).sum())),
        ("duplicate node IDs within a job", int(df["nodes_count"].ne(df["unique_nodes_count"]).sum())),
        ("allocated nodes != len(nodes)", int(df["num_nodes_alloc"].ne(df["nodes_count"]).sum())),
        ("requested nodes != allocated nodes", int(df["num_nodes_req"].ne(df["num_nodes_alloc"]).sum())),
        ("nonpositive requested nodes", int(df["num_nodes_req"].le(0).sum())),
        ("nonpositive allocated nodes", int(df["num_nodes_alloc"].le(0).sum())),
    ],
    columns=["check", "failures"],
)
display(resource_checks)

# Detect actual same-node time overlaps instead of interpreting the ambiguous
# string values in `shared` as a Boolean flag.
allocation_intervals = (
    df[["job_id", "start_time", "end_time", "nodes"]]
    .explode("nodes")
    .sort_values(["nodes", "start_time", "end_time", "job_id"])
)
overlap_job_ids = set()
overlapping_starts = 0
for _, group in allocation_intervals.groupby("nodes", sort=False):
    maximum_end = None
    maximum_end_job = None
    for job_id, start_time, end_time in group[["job_id", "start_time", "end_time"]].itertuples(index=False, name=None):
        if maximum_end is not None and start_time < maximum_end:
            overlap_job_ids.add(job_id)
            overlap_job_ids.add(maximum_end_job)
            overlapping_starts += 1
        if maximum_end is None or end_time > maximum_end:
            maximum_end = end_time
            maximum_end_job = job_id

df["observed_node_time_overlap"] = df["job_id"].isin(overlap_job_ids)
display(pd.DataFrame(
    [
        ("node-allocation interval rows", len(allocation_intervals)),
        ("intervals starting before a prior interval ended", overlapping_starts),
        ("jobs participating in at least one overlap", len(overlap_job_ids)),
    ],
    columns=["overlap metric", "value"],
))
del allocation_intervals

# Rebuild the feature catalog from the raw schema. This variable is exported
# later and its explicit availability/role labels enforce the leakage boundary.
submission_fields = {
    "cores_per_task", "num_cores_req", "num_nodes_req", "num_tasks",
    "partition", "priority", "qos", "req_nodes", "req_switch",
    "submit_time", "threads_per_core", "time_limit", "num_gpus_req", "mem_req",
}
identifier_fields = {"job_id", "group_id", "user_id"}
post_allocation_fields = {
    "cores_alloc_layout", "cores_allocated", "nodes", "num_cores_alloc",
    "num_nodes_alloc", "num_gpus_alloc", "mem_alloc",
}
oracle_fields = {"run_time", "node_power_consumption", "cpu_power_consumption", "mem_power_consumption"}
completion_fields = {"derived_ec", "end_time", "job_state", "start_time", "state_reason"}
unit_notes = {
    "run_time": "seconds", "time_limit": "minutes",
    "node_power_consumption": "whole-job W, nominal 20 s samples",
    "cpu_power_consumption": "component W, nominal 20 s samples",
    "mem_power_consumption": "component W, nominal 20 s samples",
    "mem_req": "source unit unresolved", "mem_alloc": "source unit unresolved",
}

def feature_policy(field):
    if field in submission_fields:
        return "submission", "candidate predictor / scheduler input", "low", "retain"
    if field == "eligible_time":
        return "release/eligibility", "scheduler release time", "medium at submission", "fallback only if missing"
    if field in oracle_fields:
        return "execution outcome", "oracle target / diagnostic", "high", "never use as predictor"
    if field in post_allocation_fields:
        return "post-allocation", "oracle validation", "high", "never use as initial predictor"
    if field in completion_fields:
        return "completion/outcome", "quality control / validation", "high", "never use as predictor"
    if field in identifier_fields:
        return "submission", "identifier only", "high memorization risk", "retain for joins only"
    return "unresolved", "audit only", "medium", "do not model until documented"

feature_catalog_rows = []
for field in table.schema:
    available_when, role, leakage_risk, handling = feature_policy(field.name)
    feature_catalog_rows.append(
        {
            "field": field.name,
            "actual_arrow_type": str(field.type),
            "unit_or_note": unit_notes.get(field.name, "count/category/identifier as documented"),
            "available_when": available_when,
            "initial_role": role,
        }
    )
feature_catalog = pd.DataFrame(feature_catalog_rows)
assert set(feature_catalog["field"]) == set(table.column_names)
display(feature_catalog)

df["release_time"] = df["eligible_time"].where(df["eligible_time"].notna(), df["submit_time"])
df["eligible_time_fallback_used"] = df["eligible_time"].isna()
df["component_profiles_available"] = (
    df["cpu_power_consumption_samples"].gt(0)
    & df["mem_power_consumption_samples"].gt(0)
)

rule_pass = OrderedDict(
    [
        ("completed", df["job_state"].eq("COMPLETED")),
        ("selected_partition", df["partition"].eq(SELECTED_PARTITION)),
        (
            "timestamps_ordered",
            df[TIMESTAMP_COLUMNS].notna().all(axis=1)
            & df["submit_time"].le(df["eligible_time"])
            & df["eligible_time"].le(df["start_time"])
            & df["start_time"].le(df["end_time"]),
        ),
        (
            "runtime_exact_positive",
            df["run_time"].gt(0) & df["runtime_residual_s"].eq(0),
        ),
        (
            "node_structure_valid",
            df["num_nodes_req"].gt(0)
            & df["num_nodes_alloc"].gt(0)
            & df["num_nodes_alloc"].eq(df["nodes_count"])
            & df["nodes_count"].eq(df["unique_nodes_count"]),
        ),
        ("requested_nodes_match_allocation", df["num_nodes_req"].eq(df["num_nodes_alloc"])),
        (
            "node_power_valid",
            df["node_power_consumption_samples"].gt(0)
            & df["node_power_min_W"].gt(0)
            & np.isfinite(df["node_power_mean_W"]),
        ),
        ("node_profile_within_one_sample", df["node_profile_aligned"]),
        ("no_observed_node_time_overlap", ~df["observed_node_time_overlap"]),
    ]
)

clean_mask = np.logical_and.reduce(list(rule_pass.values()))

ledger_rows = [{"stage": "raw", "rows_remaining": len(df), "rows_removed_at_stage": 0}]
running = np.ones(len(df), dtype=bool)
for rule, passed in rule_pass.items():
    before = int(running.sum())
    running &= passed.to_numpy()
    ledger_rows.append(
        {
            "stage": rule,
            "rows_remaining": int(running.sum()),
            "rows_removed_at_stage": before - int(running.sum()),
        }
    )

filter_ledger = pd.DataFrame(ledger_rows)
assert np.array_equal(running, clean_mask)
display(filter_ledger)

derived_clean_columns = [
    "release_time",
    "eligible_time_fallback_used",
    "elapsed_runtime_s",
    "node_power_consumption_samples",
    "cpu_power_consumption_samples",
    "mem_power_consumption_samples",
    "nominal_node_profile_s",
    "node_profile_delta_s",
    "node_profile_aligned",
    "node_power_mean_W",
    "node_power_min_W",
    "node_power_max_W",
    "nodes_count",
    "unique_nodes_count",
    "component_profiles_available",
    "observed_node_time_overlap",
]
clean_jobs = df.loc[clean_mask, [*table.column_names, *derived_clean_columns]].copy()


clean_assertions = OrderedDict(
    [
        ("nonempty", len(clean_jobs) > 0),
        ("unique_job_id", clean_jobs["job_id"].is_unique),
        ("completed_only", clean_jobs["job_state"].eq("COMPLETED").all()),
        ("partition_1_only", clean_jobs["partition"].eq(SELECTED_PARTITION).all()),
        ("positive_runtime", clean_jobs["run_time"].gt(0).all()),
        ("runtime_exact", clean_jobs["run_time"].eq(clean_jobs["elapsed_runtime_s"]).all()),
        ("node_request_matches_allocation", clean_jobs["num_nodes_req"].eq(clean_jobs["num_nodes_alloc"]).all()),
        ("allocated_nodes_match_list", clean_jobs["num_nodes_alloc"].eq(clean_jobs["nodes_count"]).all()),
        ("unique_nodes_within_job", clean_jobs["nodes_count"].eq(clean_jobs["unique_nodes_count"]).all()),
        ("usable_node_power", clean_jobs["node_power_min_W"].gt(0).all()),
        ("profile_within_one_sample", clean_jobs["node_profile_delta_s"].abs().le(POWER_SAMPLE_SECONDS).all()),
        ("no_observed_node_sharing", (~clean_jobs["observed_node_time_overlap"]).all()),
    ]
)
assert all(clean_assertions.values())
display(pd.Series(clean_assertions, name="passed").to_frame())

,check,failures
0,empty nodes list,0
1,duplicate node IDs within a job,0
2,allocated nodes != len(nodes),0
3,requested nodes != allocated nodes,186
4,nonpositive requested nodes,0
5,nonpositive allocated nodes,0


,overlap metric,value
0,node-allocation interval rows,974220
1,intervals starting before a prior interval ended,2763
2,jobs participating in at least one overlap,5394


,field,actual_arrow_type,unit_or_note,available_when,initial_role
0,cores_alloc_layout,string,count/category/identifier as documented,post-allocation,oracle validation
1,cores_allocated,string,count/category/identifier as documented,post-allocation,oracle validation
2,cores_per_task,int64,count/category/identifier as documented,submission,candidate predictor / scheduler input
3,derived_ec,string,count/category/identifier as documented,completion/outcome,quality control / validation
4,eligible_time,"timestamp[us, tz=UTC]",count/category/identifier as documented,release/eligibility,scheduler release time
5,end_time,"timestamp[us, tz=UTC]",count/category/identifier as documented,completion/outcome,quality control / validation
6,group_id,int64,count/category/identifier as documented,submission,identifier only
7,job_id,int64,count/category/identifier as documented,submission,identifier only
8,job_state,string,count/category/identifier as documented,completion/outcome,quality control / validation
9,nodes,list<item: int64>,count/category/identifier as documented,post-allocation,oracle validation


,stage,rows_remaining,rows_removed_at_stage
0,raw,231238,0
1,completed,180310,50928
2,selected_partition,178092,2218
3,timestamps_ordered,178092,0
4,runtime_exact_positive,178045,47
5,node_structure_valid,178045,0
6,requested_nodes_match_allocation,178008,37
7,node_power_valid,178008,0
8,node_profile_within_one_sample,161976,16032
9,no_observed_node_time_overlap,157062,4914


,passed
nonempty,True
unique_job_id,True
completed_only,True
partition_1_only,True
positive_runtime,True
runtime_exact,True
node_request_matches_allocation,True
allocated_nodes_match_list,True
unique_nodes_within_job,True
usable_node_power,True


## 7. Debug subset and exports

The debug trace is the first 5,000 clean jobs ordered by submission time and job ID.

In [17]:
debug_jobs = (
    clean_jobs.sort_values(["submit_time", "job_id"], kind="mergesort")
    .head(DEBUG_JOB_COUNT)
    .copy()
)
assert len(debug_jobs) == DEBUG_JOB_COUNT

debug_summary = pd.DataFrame(
    [
        ("jobs", len(debug_jobs)),
        ("first_submit_UTC", debug_jobs["submit_time"].min()),
        ("last_submit_UTC", debug_jobs["submit_time"].max()),
        ("calendar_span", debug_jobs["submit_time"].max() - debug_jobs["submit_time"].min()),
        ("node_power_samples", int(debug_jobs["node_power_consumption_samples"].sum())),
        ("maximum_requested_nodes", int(debug_jobs["num_nodes_req"].max())),
    ],
    columns=["property", "value"],
)
display(debug_summary)
display(debug_jobs.head(5))

output_paths = {
    "clean_dataset": OUTPUT_DIR / "pm100_clean.parquet",
    "debug_subset": OUTPUT_DIR / "pm100_debug_5000.parquet",
    "filter_ledger": OUTPUT_DIR / "pm100_filter_ledger.csv",
    "feature_catalog": OUTPUT_DIR / "pm100_feature_catalog.csv",
    "manifest": OUTPUT_DIR / "pm100_manifest.json",
}

manifest = {
    "phase": 0,
    "status": "complete",
    "source": {
        "path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "rows": len(df),
        "columns": len(table.column_names),
    },
    "decisions": {
        "selected_partition": SELECTED_PARTITION,
        "included_state": "COMPLETED",
        "power_unit": "W",
        "power_aggregation": "whole-job sum across allocated nodes and both power sockets",
        "nominal_sample_interval_seconds": POWER_SAMPLE_SECONDS,
        "profile_tolerance_seconds": POWER_SAMPLE_SECONDS,
        "node_request_must_equal_allocation": True,
        "observed_node_time_overlap_allowed": False,
        "component_power_missing_policy": "flag",
    },
    "outputs": {
        "clean_rows": len(clean_jobs),
        "debug_rows": len(debug_jobs),
        "files": {name: str(path.relative_to(PROJECT_ROOT)) for name, path in output_paths.items()},
    },
}

if WRITE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    clean_jobs.to_parquet(output_paths["clean_dataset"], index=False)
    debug_jobs.to_parquet(output_paths["debug_subset"], index=False)
    filter_ledger.to_csv(output_paths["filter_ledger"], index=False)
    feature_catalog.to_csv(output_paths["feature_catalog"], index=False)
    output_paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str) + "\n")
else:
    display(Markdown("`WRITE_OUTPUTS=False`: analysis completed without writing processed artifacts."))

,property,value
0,jobs,5000
1,first_submit_UTC,2020-05-06 07:04:53+00:00
2,last_submit_UTC,2020-05-20 23:13:55+00:00
3,calendar_span,14 days 16:09:02
4,node_power_samples,179818
5,maximum_requested_nodes,256


,cores_alloc_layout,cores_allocated,cores_per_task,derived_ec,eligible_time,end_time,group_id,job_id,job_state,nodes,num_cores_req,num_cores_alloc,num_nodes_req,num_nodes_alloc,num_tasks,partition,priority,qos,req_nodes,req_switch,run_time,shared,start_time,state_reason,submit_time,threads_per_core,time_limit,num_gpus_req,num_gpus_alloc,mem_req,mem_alloc,user_id,node_power_consumption,mem_power_consumption,cpu_power_consumption,release_time,eligible_time_fallback_used,elapsed_runtime_s,node_power_consumption_samples,cpu_power_consumption_samples,mem_power_consumption_samples,nominal_node_profile_s,node_profile_delta_s,node_profile_aligned,node_power_mean_W,node_power_min_W,node_power_max_W,nodes_count,unique_nodes_count,component_profiles_available,observed_node_time_overlap
8272,"{332: [0, 1, 2, 3, 4, 5, 6, 7, 16, 17, 18, 19, 20, 21, 22, 23], 320: [0, 1, 2, 3, 4, 5, 6, 7, 16...","{332: 64, 320: 64, 321: 64, 322: 64, 323: 64, 324: 64, 325: 64, 326: 64, 327: 64, 328: 64, 333: ...",16,0:0,2020-05-06 07:04:53+00:00,2020-05-06 11:50:09+00:00,25200,2214444,COMPLETED,"[320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 331, 332, 333]",832,832,13,13,NaN,1,86732,1,NaN,0,17052,OK,2020-05-06 07:05:57+00:00,None,2020-05-06 07:04:53+00:00,NaN,1440,52,52,3123,3123,303,"[8800, 690, 8380, 6240, 8420, 5810, 8520, 3830, 8590, 5140, 8320, 5690, 8330, 3180, 8330, 6810, ...","[472, 144, 482, 256, 468, 298, 474, 76, 478, 330, 468, 324, 478, 180, 472, 474, 476, 266, 472, 2...","[1604, 752, 1890, 740, 1342, 782, 1476, 208, 1316, 932, 1318, 904, 1292, 498, 1300, 1336, 1302, ...",2020-05-06 07:04:53+00:00,False,17052,852,853,853,17040,-12,True,"6,935.692",540,12000,13,13,True,False
8210,"{150: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,...","{150: 128, 149: 128, 152: 128, 151: 128, 154: 128, 153: 128, 156: 128, 148: 128, 155: 128}",1,0:0,2020-05-06 07:59:15+00:00,2020-05-06 10:02:31+00:00,25200,4281751,COMPLETED,"[148, 149, 150, 151, 152, 153, 154, 155, 156]",1152,1152,9,9,NaN,1,302763,1,NaN,0,7342,OK,2020-05-06 08:00:09+00:00,None,2020-05-06 07:59:15+00:00,NaN,720,36,36,576,576,1149,"[7630, 2520, 7700, 3390, 7150, 5590, 5110, 2180, 4820, 4310, 4800, 4810, 4780, 1610, 4770, 3720,...","[36, 562, 330, 566, 568, 730, 258, 510, 552, 674, 538, 690, 380, 542, 316, 602, 544, 618, 212, 5...","[300, 3012, 1400, 2666, 2648, 2598, 1248, 2812, 2496, 2580, 2092, 2508, 1584, 2602, 2084, 2616, ...",2020-05-06 07:59:15+00:00,False,7342,367,366,366,7340,-2,True,"4,890.027",640,9690,9,9,True,False
8302,"{676: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,...","{676: 128, 679: 128, 677: 128, 678: 128}",32,0:0,2020-05-06 08:04:51+00:00,2020-05-06 09:58:40+00:00,25200,584948,COMPLETED,"[676, 677, 678, 679]",512,512,4,4,NaN,1,262052,1,NaN,0,6749,OK,2020-05-06 08:06:11+00:00,None,2020-05-06 08:04:51+00:00,NaN,240,16,16,920,920,553,"[2200, 1650, 2190, 2200, 2210, 1650, 2200, 2200, 2200, 1670, 2210, 2200, 2190, 540, 2200, 1640, ...","[72, 146, 110, 152, 112, 416, 104, 144, 144, 144, 72, 146, 110, 144, 72, 144, 110, 146, 144, 144...","[222, 470, 356, 454, 336, 1266, 308, 470, 450, 438, 230, 432, 314, 430, 218, 452, 308, 430, 430,...",2020-05-06 08:04:51+00:00,False,6749,338,329,329,6760,11,True,"2,282.574",540,3610,4,4,True,False
7962,"{613: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,...","{613: 128, 614: 128, 615: 128, 616: 128, 617: 128, 618: 128}",32,0:15,2020-05-06 08:31:39+00:00,2020-05-06 10:53:57+00:00,25200,3096661,COMPLETED,"[613, 614, 615, 616, 617, 618]",768,768,6,6,NaN,1,262036,1,NaN,0,8499,OK,2020-05-06 08:32:18+00:00,None,2020-05-06 08:31:39+00:00,NaN,240,24,24,1380,1380,553,"[5390, 3570, 5420, 5430, 5390, 3590, 5220, 3240, 4850, 4970, 5020, 3320, 5080, 3470, 5140, 1720,...","[40, 216, 72, 222, 110, 222, 144, 216, 72, 220, 36, 216, 72, 218, 72, 224, 108, 220, 110, 224, 1...","[134, 1062, 586, 1348, 340, 1464, 658, 1410, 532, 134